# Tie-break ablation (A2)

Rank the same scores under three operators and measure the disagreement. Because
all three consume **bit-identical** scores, any disagreement is caused by the
tie-break and by nothing else.

In [ ]:
import sys
from pathlib import Path

# Run from anywhere: notebooks/ is a sibling of src/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

In [ ]:
from tfidf_stability.analysis.tie_break_ablations import ablate_queries, disagreement_rate
from tfidf_stability.datasets.loaders import load_dataset
from tfidf_stability.preprocessing.pipeline import PreprocessingPipeline
from tfidf_stability.ranking.attributes import AttributeTable
from tfidf_stability.ranking.ranker import rank_all_operators
from tfidf_stability.similarity.cosine import cosine_against_corpus
from tfidf_stability.utils.numerics import same_bits
from tfidf_stability.vectorisation.tfidf import TfidfVectoriser

data = load_dataset("synthetic_tiny")
pipeline = PreprocessingPipeline()
features = [pipeline.preprocess(str(r["text"])) for r in data.records]
model = TfidfVectoriser().fit(features, data.doc_ids)
table = AttributeTable.from_records(data.records)
documents = [model.document(i) for i in range(model.n_documents)]

scores_by_query = [
    cosine_against_corpus(
        TfidfVectoriser.transform_query(list(f)[:6], model), documents, model.norms)
    for f in features[::4]
]

# A2's premise: every operator consumes the same scores.
#
# Compared on `ranking.scores`, which is a distinct object per operator.
# `rank_all_operators` shares one `sorted_scores` array across operators, so
# comparing that array between them compares an object with itself.
compared = 0
divergences = []
for query_index, scores in enumerate(scores_by_query):
    rankings = rank_all_operators(scores, table)
    reference = rankings["pi"]
    for name, ranking in rankings.items():
        for position, (mine, theirs) in enumerate(
            zip(ranking.scores, reference.scores, strict=True)
        ):
            if not same_bits(mine, theirs):
                divergences.append((query_index, name, position, mine, theirs))
            compared += 1

assert compared > 0, "the premise check compared nothing"
assert not divergences, (
    f"{len(divergences)} score divergences, e.g. query {divergences[0][0]} "
    f"operator {divergences[0][1]} position {divergences[0][2]}: "
    f"{divergences[0][3]!r} != {divergences[0][4]!r}. A2's premise is false and "
    f"no disagreement rate below is attributable to the tie-break."
)

# One call, so the identity holds within a single set of results.
one_run = rank_all_operators(scores_by_query[0], table)
shared = all(r.sorted_scores is one_run["pi"].sorted_scores for r in one_run.values())
print(f"same_bits comparisons of raw scores against pi, all passing: {compared}")
print(f"operators also share one sorted-score object: {shared}")

In [ ]:
ks = tuple(k for k in (1, 5, 10, 20, 50) if k < model.n_documents)
results = ablate_queries([(f"q{i}", s) for i, s in enumerate(scores_by_query)], table, ks=ks)

pairs = sorted({(p.baseline, p.variant) for r in results for p in r.pairs})
for baseline, variant in pairs:
    cells = []
    for k in ks:
        rate, n = disagreement_rate(results, baseline, variant, k)
        cells.append(f"k{k}={rate:5.1%}(n={n})")
    print(f"{baseline:9} vs {variant:9}  " + "  ".join(cells))

Disagreement concentrates at **k=1**: the top document frequently sits in an
exact-tie block, so which one is returned first is decided entirely by the
tie-break. That is the decision-level discontinuity A2 names, at the rank where a
recommender's output is most visible.

Every rate carries its denominator. A rate without its `n` cannot be
distinguished from noise over three queries.